# Baseline Verification: All Exercises × MLP
Runs binary verification for all baseline MLP models across portscan and dos_http_flood properties.
RF models are skipped (cannot be exported to ONNX for Marabou).
Structure: `baselines/{ex}/{mlp}/{run}/model.joblib`

In [1]:
import sys
import warnings
from pathlib import Path
import torch
import torch.nn as nn
import joblib
import numpy as np
import json
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path(".").resolve()))
sys.path.insert(0, str(Path("../Prop_repo/Property-based-Neural-Network-Verification/src").resolve()))
sys.path.insert(0, "/Users/karlivarkvaran/Desktop/Attacks_on_models/Marabou")
sys.path.insert(0, "/Users/karlivarkvaran/Desktop/Attacks_on_models/Marabou/build")

import utils.models

class MLP(nn.Module):
    def __init__(self, n_features: int, num_classes: int):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 64),         nn.ReLU(),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        x = x.squeeze(1)
        return self.head(x)

utils.models.MLP = MLP

class FlatMLPWrapper(nn.Module):
    def __init__(self, mlp):
        super().__init__()
        self.mlp = mlp
    def forward(self, x):
        return self.mlp(x)

from maraboupy.MarabouNetworkONNX import MarabouNetworkONNX
from maraboupy import Marabou, MarabouCore
from maraboupy.MarabouUtils import Equation as MarabouEquation
EQ_LE = MarabouCore.Equation.LE

print("\u2713 All imports successful")

✓ All imports successful


In [ ]:
# UTILITY FUNCTIONS

def make_sb(bundle):
    scaler = bundle.get("scaler")
    scale_cols = list(bundle.get("scale_cols", []))

    def sb(feature, raw_val):
        if scaler is None or feature not in scale_cols:
            return float(raw_val)
        i = scale_cols.index(feature)
        if hasattr(scaler, "data_min_"):
            return float(np.clip(
                (raw_val - scaler.data_min_[i]) / scaler.data_range_[i], 0.0, 1.0
            ))
        if hasattr(scaler, "mean_"):
            return float((raw_val - scaler.mean_[i]) / scaler.scale_[i])
        return float(raw_val)
    return sb

def export_onnx(bundle, path):
    model = bundle["model"].eval()
    n_feat = len(bundle["features"])
    wrapped = FlatMLPWrapper(model).eval()
    dummy = torch.randn(1, n_feat)
    torch.onnx.export(wrapped, dummy, str(path),
        input_names=["input"], output_names=["output"],
        opset_version=11, dynamo=False)

def _norm(s):
    return s.upper().replace("-", "_").replace(" ", "_")

def find_label_idx(labels, candidates):
    normed = [_norm(l) for l in labels]
    for c in candidates:
        cn = _norm(c)
        if cn in normed:
            return normed.index(cn)
    return None

ATTACK_CANDIDATES = {
    "portscan":       ["portscan", "PORTSCAN", "port_scan", "attack", "ATTACK", "malicious"],
    "dos_http_flood": ["dos_http_flood", "DOS_HTTP_FLOOD", "http_flood", "HTTP_FLOOD",
                       "dos_http", "attack", "ATTACK", "malicious"],
}
BENIGN_CANDIDATES = ["benign", "BENIGN", "normal", "NORMAL", "background"]

RAW_DATA_BOUNDS = {
    "duration": (1.0, 180.0), "orig_bytes": (0.0, 500_000.0), "orig_ip_bytes": (0.0, 500_000.0),
    "resp_pkts": (0.0, 1_000.0), "resp_bytes": (0.0, 500_000.0), "resp_ip_bytes": (0.0, 500_000.0),
    "missed_bytes": (0.0, 1_000.0), "orig_pkts": (0.0, 10_000.0), "orig_pkt_rate": (0.0, 10_000.0),
    "orig_byte_rate": (0.0, 1_000_000.0), "pkts_per_port": (0.0, 10_000.0), "time_elapsed": (0.0, 120.0),
    "uniq_dst_ports": (0.0, 999.0), "scan_duration": (0.0, 120.0),
    "valid_tcp_handshake": (0.0, 1.0), "valid_http_conn": (0.0, 1.0), "fail_ratio": (0.0, 1.0),
    "proto": (0.0, 10.0), "service": (0.0, 20.0), "conn_state": (0.0, 20.0), "history": (0.0, 100.0),
}

PORTSCAN_SPECS = {
    'uniq_dst_ports':     (100.0,  966.0),
    'pkts_per_port':      (1.02,   5.0),
    'scan_duration':      (0.02,   10.0),
    'fail_ratio':         (0.75,   1.0),
    'orig_bytes':         (0.0,    0.0),
    'resp_bytes':         (0.0,    0.0),
    'missed_bytes':       (0.0,    0.0),
    'orig_pkts':          (1.0,    1.0),
    'resp_pkts':          (1.0,    1.0),
    'valid_tcp_handshake':(0.0,    0.0),
    'valid_http_conn':    (0.0,    0.0),
    'proto':              (1.0,    1.0),
    'service':            (0.0,    0.0),
    'conn_state':         (5.0,    5.0),
    'history':            (37.0,   37.0),
    'duration':           (0.0,    9.0),
    'orig_ip_bytes':      (44.0,   383.0),
    'resp_ip_bytes':      (0.0,    2986.0),
    'time_elapsed':       (0.0,    215.0),
}

DOS_HTTP_FLOOD_SPECS = {
    'orig_bytes':         (250.0,  4848.0),
    'orig_ip_bytes':      (567.0,  6767.0),
    'time_elapsed':       (0.0,    100.0),
    'orig_pkts':          (5.0,    12.0),
    'pkts_per_port':      (2104.0, 19000.0),
    'fail_ratio':         (0.53,   1.0),
    'scan_duration':      (1.0,    222.0),
    'duration':           (0.01,   2.0),
    'resp_pkts':          (3.0,    10.0),
    'uniq_dst_ports':     (1.0,    1.0),
    'resp_bytes':         (11595.0,11595.0),
    'missed_bytes':       (0.0,    0.0),
    'valid_http_conn':    (1.0,    1.0),
    'valid_tcp_handshake':(1.0,    1.0),
    'proto':              (1.0,    1.0),
    'service':            (1.0,    1.0),
    'conn_state':         (4.0,    4.0),
    'history':            (37.0,   37.0),
    'resp_ip_bytes':      (11863.0,12019.0),
}

def build_property_bounds(bundle, attack_specs):
    sb = make_sb(bundle)
    scale_cols = set(bundle.get("scale_cols", []))
    bounds = {}
    for feat in bundle["features"]:
        raw_lo, raw_hi = attack_specs.get(feat, RAW_DATA_BOUNDS.get(feat, (0.0, 1.0)))
        if feat in scale_cols:
            bounds[feat] = (sb(feat, raw_lo), sb(feat, raw_hi))
        else:
            bounds[feat] = (float(raw_lo), float(raw_hi))
    return bounds

EPS = 0.5

def add_relational_constraints(net, bundle, property_name="portscan"):
    features = list(bundle["features"])
    inputs = net.inputVars[0][0]
    scale_cols = list(bundle["scale_cols"])
    scaler = bundle["scaler"]

    def get_range(feat):
        if feat in scale_cols:
            return float(scaler.data_range_[scale_cols.index(feat)])
        return 1.0

    def get_min(feat):
        if feat in scale_cols:
            return float(scaler.data_min_[scale_cols.index(feat)])
        return 0.0

    def feat_idx(feat):
        return features.index(feat)

    def add_ineq(feat_a, coeff_a, feat_b, coeff_b):
        if feat_a not in features or feat_b not in features:
            return
        range_a, range_b = get_range(feat_a), get_range(feat_b)
        min_a, min_b = get_min(feat_a), get_min(feat_b)
        rhs = -(coeff_a * min_a + coeff_b * min_b) / range_a
        eq = MarabouEquation(EQ_LE)
        eq.addAddend(coeff_a, inputs[feat_idx(feat_a)])
        eq.addAddend(coeff_b * range_b / range_a, inputs[feat_idx(feat_b)])
        eq.setScalar(rhs)
        net.addEquation(eq)

    add_ineq("orig_bytes", 1.0, "orig_ip_bytes", -1.0)
    add_ineq("resp_bytes", 1.0, "resp_ip_bytes", -1.0)

    if "orig_pkts" in features and "orig_bytes" in features:
        range_ob = get_range("orig_bytes")
        range_op = get_range("orig_pkts")
        min_op = get_min("orig_pkts")
        min_ob = get_min("orig_bytes")
        eq = MarabouEquation(EQ_LE)
        eq.addAddend(40.0 * range_op / range_ob, inputs[feat_idx("orig_pkts")])
        eq.addAddend(-1.0, inputs[feat_idx("orig_bytes")])
        eq.setScalar(-(40.0 * min_op - min_ob) / range_ob)
        net.addEquation(eq)

def _verify_one_rival(onnx_path, bundle, bounds, attack_idx, rival_idx, timeout_s=120):
    net = MarabouNetworkONNX(onnx_path)
    features = bundle["features"]

    for i, feat in enumerate(features):
        lo, hi = bounds.get(feat, (0.0, 1.0))
        net.setLowerBound(net.inputVars[0][0][i], lo)
        net.setUpperBound(net.inputVars[0][0][i], hi)

    add_relational_constraints(net, bundle)

    out_vars = net.outputVars[0][0]
    eq = MarabouEquation(EQ_LE)
    eq.addAddend(1.0,  out_vars[attack_idx])
    eq.addAddend(-1.0, out_vars[rival_idx])
    eq.setScalar(-EPS)
    net.addEquation(eq)

    options = Marabou.createOptions(verbosity=0, timeoutInSeconds=timeout_s)
    exitCode, vals, stats = net.solve(options=options)

    if stats.hasTimedOut():
        return "TIMEOUT", None
    return ("UNSAT", None) if len(vals) == 0 else ("SAT", vals)

def verify_multiclass(onnx_path, bundle, bounds, attack_class_key, timeout_s=120):
    labels = bundle["labels"]
    attack_idx = find_label_idx(labels, ATTACK_CANDIDATES.get(attack_class_key, [attack_class_key]))
    if attack_idx is None:
        return "SKIP", {}

    overall = "UNSAT"
    rivals = {}
    for rival_idx, rival_name in enumerate(labels):
        if rival_idx == attack_idx:
            continue
        result, _ = _verify_one_rival(onnx_path, bundle, bounds, attack_idx, rival_idx, timeout_s)
        rivals[rival_name] = result
        if result == "SAT":
            overall = "SAT"
        elif result == "TIMEOUT" and overall != "SAT":
            overall = "TIMEOUT"

    return overall, rivals

def compute_cacc(bundle, prop_name, label_key, n_samples=200):
    try:
        specs = PORTSCAN_SPECS if prop_name == "portscan" else DOS_HTTP_FLOOD_SPECS
        bounds = build_property_bounds(bundle, specs)
        X = []
        for _ in range(n_samples):
            x = [np.random.uniform(bounds[feat][0], bounds[feat][1]) for feat in bundle["features"]]
            X.append(x)

        model = bundle.get("model")
        if model is None:
            return None, 0
        model.eval()
        with torch.no_grad():
            inp = torch.tensor(X, dtype=torch.float32)
            out = model(inp).numpy()

        attack_idx = find_label_idx(bundle["labels"], ATTACK_CANDIDATES.get(label_key, [label_key]))
        if attack_idx is None:
            return None, 0

        attack_wins = np.all(
            out[:, attack_idx:attack_idx+1] > np.delete(out, attack_idx, axis=1),
            axis=1
        )
        cacc = float(attack_wins.sum()) / n_samples
        return cacc, n_samples
    except Exception:
        return None, 0

print("\u2713 Utility functions defined")

✓ Utility functions defined


In [ ]:
# Quick sanity check :)
bundle_path = next(Path("baselines").glob("*/mlp/*/model.joblib"), None)
print(f"Loading: {bundle_path}\n")
bundle = joblib.load(bundle_path)

print(f"Labels:     {bundle['labels']}")
print(f"Features:   {list(bundle['features'])}")
print(f"Model type: {bundle.get('model_type', 'unknown')}")

scaler = bundle.get("scaler")
scale_cols = list(bundle.get("scale_cols", []))
if scaler is not None:
    print(f"\nScaler: {type(scaler).__name__}")
    print(f"\n{'Feature':<25} {'data_min':>14} {'data_max':>14} {'data_range':>14}")
    print("-" * 72)
    for i, col in enumerate(scale_cols):
        d_min = scaler.data_min_[i]
        d_rng = scaler.data_range_[i]
        print(f"{col:<25} {d_min:>14.4f} {d_min + d_rng:>14.4f} {d_rng:>14.4f}")

Loading: baselines/ex1/mlp/2026-05-22_232919_run2/model.joblib

Labels:     ['BENIGN', 'ATTACK']
Features:   ['duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'time_elapsed', 'valid_tcp_handshake', 'valid_http_conn', 'uniq_dst_ports', 'pkts_per_port', 'scan_duration', 'fail_ratio']
Model type: mlp

Scaler: MinMaxScaler

Feature                         data_min       data_max     data_range
------------------------------------------------------------------------
duration                          0.0000       180.1099       180.1099
orig_bytes                        0.0000      4891.3100      4891.3100
resp_bytes                        0.0000     57537.9300     57537.9300
missed_bytes                      0.0000     11458.4800     11458.4800
orig_pkts                         1.0000        46.0000        45.0000
orig_ip_bytes                    44.0000      6799.6200      6755.6200
resp_pkts                         0.0000 

In [ ]:
def get_all_baseline_mlp_paths():
    #Discover all baseline MLP models: baselines/{ex}/mlp/{run}/model.joblib
    base = Path("baselines")
    models = []

    for ex_dir in sorted(base.iterdir()):
        if not ex_dir.is_dir() or not ex_dir.name.startswith("ex"):
            continue
        exercise = ex_dir.name

        mlp_dir = ex_dir / "mlp"
        if not mlp_dir.exists():
            continue

        for run_dir in sorted(mlp_dir.iterdir()):
            if not run_dir.is_dir():
                continue
            model_path = run_dir / "model.joblib"
            if model_path.exists():
                models.append({
                    "exercise": exercise,
                    "model_type": "mlp",
                    "run": run_dir.name,
                    "path": model_path,
                })

    return models

models = get_all_baseline_mlp_paths()
print(f"Found {len(models)} baseline MLP models to verify")
print(f"Expected: 4 exercises × 5 runs = 20")
print()
for m in models:
    print(f"  {m['exercise']}/mlp/{m['run']}")

Found 19 baseline MLP models to verify
Expected: 4 exercises × 5 runs = 20

  ex1/mlp/2026-05-22_231949_run1
  ex1/mlp/2026-05-22_232919_run2
  ex1/mlp/2026-05-22_233453_run3
  ex1/mlp/2026-05-22_234635_run4
  ex2/mlp/2026-05-23_103113_run1
  ex2/mlp/2026-05-23_103630_run2
  ex2/mlp/2026-05-23_103924_run3
  ex2/mlp/2026-05-23_104153_run4
  ex2/mlp/2026-05-23_104916_run5
  ex3/mlp/2026-05-23_105218_run1
  ex3/mlp/2026-05-23_105643_run2
  ex3/mlp/2026-05-23_110140_run3
  ex3/mlp/2026-05-23_110541_run4
  ex3/mlp/2026-05-23_111150_run5
  ex4/mlp/2026-05-23_111638_run1
  ex4/mlp/2026-05-23_112525_run2
  ex4/mlp/2026-05-23_113219_run3
  ex4/mlp/2026-05-23_113617_run4
  ex4/mlp/2026-05-23_113925_run5


In [ ]:
#Main verification loop

results_all = {}
cacc_results = {}
timing_results = {}
results_by_exercise = defaultdict(lambda: {"UNSAT": 0, "SAT": 0, "TIMEOUT": 0, "ERROR": 0})

start_time = datetime.now()

for idx, model_info in enumerate(models, 1):
    key = f"{model_info['exercise']}/mlp/{model_info['run']}"
    model_start = datetime.now()

    try:
        bundle = joblib.load(str(model_info["path"]))

        onnx_path = f"/tmp/baseline_{model_info['exercise']}_{model_info['run']}.onnx"
        export_onnx(bundle, onnx_path)

        results_all[key]  = {}
        cacc_results[key] = {}
        timing_results[key] = {}

        for prop_name, specs in [("portscan", PORTSCAN_SPECS), ("dos_http_flood", DOS_HTTP_FLOOD_SPECS)]:
            prop_start = datetime.now()

            bounds = build_property_bounds(bundle, specs)
            overall, rivals = verify_multiclass(onnx_path, bundle, bounds, prop_name, timeout_s=120)
            results_all[key][prop_name] = {"overall": overall, "rivals": rivals}

            cacc, n_region = compute_cacc(bundle, prop_name, prop_name)
            cacc_results[key][prop_name] = {"cacc": cacc, "n_region": n_region}

            prop_time = (datetime.now() - prop_start).total_seconds()
            timing_results[key][prop_name] = prop_time

            if overall != "SKIP":
                results_by_exercise[model_info['exercise']][overall] += 1

        model_time = (datetime.now() - model_start).total_seconds() / 60
        elapsed    = (datetime.now() - start_time).total_seconds() / 60
        rate       = idx / elapsed if elapsed > 0 else 0
        remaining  = (len(models) - idx) / rate if rate > 0 else 0

        cacc_ps  = cacc_results[key].get("portscan", {}).get("cacc")
        cacc_dos = cacc_results[key].get("dos_http_flood", {}).get("cacc")
        ov_ps    = results_all[key]["portscan"]["overall"]
        ov_dos   = results_all[key]["dos_http_flood"]["overall"]
        rivals_ps  = results_all[key]["portscan"].get("rivals", {})
        rivals_dos = results_all[key]["dos_http_flood"].get("rivals", {})
        cacc_str = f"PS:{cacc_ps:.2f}({ov_ps}) DOS:{cacc_dos:.2f}({ov_dos})" if cacc_ps is not None else "N/A"
        rival_str = f"  rivals_ps={rivals_ps} rivals_dos={rivals_dos}"
        print(f"[{idx:2d}/{len(models)}] {key:40s}  {cacc_str}  {model_time:.1f}m  ETA:{remaining:.0f}m")
        print(f"          {rival_str}")

    except Exception as e:
        print(f"[{idx:2d}/{len(models)}] {key:40s}  ERROR: {str(e)[:80]}")
        results_all[key] = {"error": str(e)}
        results_by_exercise[model_info['exercise']]["ERROR"] += 2

print(f"\n{'='*80}")
print(f"\u2713 Verification complete")
elapsed_total = (datetime.now() - start_time).total_seconds() / 3600
print(f"Total time: {elapsed_total:.1f} hours")

unsat
sat
input 0 = 0.011105449853240044
input 1 = 0.5045420140540768
input 2 = 0.20064219345264483
input 3 = 0.0
input 4 = 0.24444444444444444
input 5 = 0.7807292242354524
input 6 = 0.1883496900272497
input 7 = 0.199344033901257
input 8 = 0.049766258017313915
input 9 = 1.0
input 10 = 1.0
input 11 = 0.0
input 12 = 0.3125302222138999
input 13 = 0.0012725777584355862
input 14 = 0.53
output 0 = 1.8450497845747702
output 1 = -1.6310672879515837
[ 1/19] ex1/mlp/2026-05-22_231949_run1            PS:1.00(UNSAT) DOS:0.79(SAT)  0.0m  ETA:0m
            rivals_ps={'BENIGN': 'UNSAT'} rivals_dos={'BENIGN': 'SAT'}
unsat
sat
input 0 = 5.552167020212793e-05
input 1 = 0.25469872007494876
input 2 = 0.20151924130742974
input 3 = 0.0
input 4 = 0.24444444444444444
input 5 = 0.39255096029533615
input 6 = 0.12051510042694448
input 7 = 0.1979723809921152
input 8 = 0.050154788889847327
input 9 = 1.0
input 10 = 1.0
input 11 = 0.0
input 12 = 0.18876146177536268
input 13 = 0.2744409003164611
input 14 = 0.53
outp

In [6]:
# Save all results: verification, cacc, and timing
output_file = f"baseline_verification_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
output_data = {
    "verification": results_all,
    "cacc": cacc_results,
    "timing": timing_results,
}
with open(output_file, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"\u2713 Results saved to {output_file}")
print(f"  - Verification results: {len(results_all)} models")
print(f"  - Cacc scores: {len(cacc_results)} models")
print(f"  - Timing data: {len(timing_results)} models")

✓ Results saved to baseline_verification_results_20260524_211857.json
  - Verification results: 19 models
  - Cacc scores: 19 models
  - Timing data: 19 models


In [ ]:
# Summary tables
import pandas as pd

print("\n" + "="*80)
print("RESULTS BY EXERCISE (MLP baselines)")
print("="*80)
df_ex = pd.DataFrame(results_by_exercise).T
print(df_ex)

print("\n" + "="*100)
print("PER-RUN DETAIL")
print("="*100)
rows = []
for key, props in sorted(results_all.items()):
    if "error" in props:
        rows.append({"key": key, "ps_overall": "ERROR", "dos_overall": "ERROR",
                     "ps_cacc": None, "dos_cacc": None})
        continue
    ps = props.get("portscan", {})
    dos = props.get("dos_http_flood", {})
    ps_cacc = cacc_results.get(key, {}).get("portscan", {}).get("cacc")
    dos_cacc = cacc_results.get(key, {}).get("dos_http_flood", {}).get("cacc")
    rows.append({
        "key": key,
        "ps_overall": ps.get("overall", "?"),
        "dos_overall": dos.get("overall", "?"),
        "ps_cacc": f"{ps_cacc:.3f}" if ps_cacc is not None else "N/A",
        "dos_cacc": f"{dos_cacc:.3f}" if dos_cacc is not None else "N/A",
        "ps_rivals": str(ps.get("rivals", {})),
        "dos_rivals": str(dos.get("rivals", {})),
    })

df_detail = pd.DataFrame(rows).set_index("key")
print(df_detail.to_string())

print("\n" + "="*100)
print("PER-RIVAL SAT/UNSAT BREAKDOWN (ex3 and ex4 only)")
print("="*100)
for key, props in sorted(results_all.items()):
    if "error" in props:
        continue
    ex = key.split("/")[0]
    if ex not in ("ex3", "ex4"):
        continue
    for prop_name in ("portscan", "dos_http_flood"):
        if prop_name not in props:
            continue
        overall = props[prop_name].get("overall", "?")
        rivals  = props[prop_name].get("rivals", {})
        rival_str = "  ".join(f"{r}:{v}" for r, v in rivals.items())
        print(f"  {key}/{prop_name:15s}  overall={overall}  {rival_str}")


RESULTS BY EXERCISE (MLP baselines)
     UNSAT  SAT  TIMEOUT  ERROR
ex1      4    4        0      0
ex2      5    5        0      0
ex3      6    4        0      0
ex4      5    5        0      0

PER-RUN DETAIL
                               ps_overall dos_overall ps_cacc dos_cacc                                                          ps_rivals                                                 dos_rivals
key                                                                                                                                                                                                 
ex1/mlp/2026-05-22_231949_run1      UNSAT         SAT   1.000    0.810                                                {'BENIGN': 'UNSAT'}                                          {'BENIGN': 'SAT'}
ex1/mlp/2026-05-22_232919_run2      UNSAT         SAT   1.000    0.905                                                {'BENIGN': 'UNSAT'}                                          {'BENIGN': 'SAT'}